# 07. Pruebas con los otros LLM

Se comparan **Command R7B, LFM2.5, Llama 3.1, DeepSeek-R1, gpt-oss y Mistral Small** con los ocho casos de `casos2.json`, en español e inglés (**96 pruebas**).

Las respuestas se agregan a `resultados.csv`, sin borrar los registros anteriores.

## 1. Cargar los casos, los modelos y resultados previos

In [1]:
import os
import csv
import json
import time
from copy import deepcopy

import requests

archivo_resultados = "resultados.csv"
url_ollama = "http://localhost:11434/api"
num_ctx = 8192
num_predict = 4096

columnas = [
    "id_caso", "id_modelo", "idioma", "intento", "solicitud_json", "estado",
    "tiempo_s", "respuesta", "error", "evaluacion", "motivo_evaluacion"
]

# Cargar los ocho casos
with open("casos2.json", "r", encoding="utf-8-sig") as f:
    datos = json.load(f)

casos = {c["id_caso"]: c for c in datos["casos"]}
ids_casos = [f"C{i:03d}" for i in range(1, 9)]
idiomas = ["es", "en"]

# Probar los seis modelos restantes, uno a la vez
ids_modelos = ["M14", "M22", "M09", "M10", "M17", "M18"]
with open("inventario_22_llm.csv", "r", encoding="utf-8-sig", newline="") as f:
    modelos_por_id = {m["id_modelo"]: m for m in csv.DictReader(f)}

modelos_a_probar = [modelos_por_id[i] for i in ids_modelos]

# Leer resultados previos o crear el CSV si no existe
if os.path.exists(archivo_resultados):
    with open(archivo_resultados, "r", encoding="utf-8-sig", newline="") as f:
        registros = list(csv.DictReader(f))
else:
    with open(archivo_resultados, "w", encoding="utf-8-sig", newline="") as f:
        csv.DictWriter(f, fieldnames=columnas).writeheader()
    registros = []

# Guardar qué pruebas ya quedaron completas
completadas = set()
for fila in registros:
    if fila["estado"] == "completa":
        completadas.add((fila["id_modelo"], fila["id_caso"], fila["idioma"]))

for m in modelos_a_probar:
    print(m["id_modelo"], "|", m["nombre_modelo"])
print("Casos:", len(ids_casos), "| Idiomas:", len(idiomas))
print("Pruebas previstas:", len(modelos_a_probar) * len(ids_casos) * len(idiomas))
print("Registros anteriores:", len(registros))

M14 | Command R7B 12-2024
M22 | LFM2.5 8B A1B
M09 | Llama 3.1 8B Instruct
M10 | DeepSeek-R1 0528 Qwen3 8B
M17 | gpt-oss 20B
M18 | Mistral Small 3.2 24B Instruct 2506
Casos: 8 | Idiomas: 2
Pruebas previstas: 96
Registros anteriores: 264


## 2. Preparar las solicitudes

Todos los modelos reciben la predicción de coautoría y los textos de las publicaciones en español e inglés.

In [2]:
instrucciones = """Redacta una justificación semántica breve de la predicción de coautoría recibida, con un máximo de 180 palabras.

Responde únicamente en tres apartados:

1. Predicción: indica la clasificación recibida. Si hay probabilidad y umbral, menciona sus valores.
2. Coincidencias y diferencias: explica los temas o métodos de las publicaciones de ambos autores y cita sus artículos.
3. Limitaciones: indica qué información falta o qué afirmaciones no pueden respaldarse.

Reglas:
- La predicción ya fue calculada. No intentes explicar cómo la obtuvo el predictor.
- Utiliza la clasificación conjunta cuando exista; en caso contrario, utiliza la clasificación por umbral indicada para el método.
- Si faltan probabilidades, indícalo sin dejar de comparar las publicaciones.
- La similitud entre artículos no es una probabilidad de coautoría.
- Una similitud indefinida no significa que no existan coincidencias.
- No inventes información ni afirmes que las coincidencias causaron la predicción.
- No repitas las instrucciones ni copies los resúmenes completos."""

def preparar_solicitud(caso, idioma, etiqueta):
    entrada = deepcopy(caso["entrada_llm"])
    umbral_general = datos["regla_coautoria"].get("umbral_definitivo")

    for metodo in entrada["prediccion_coautoria"]["resultados_por_metodo"]:
        p = metodo.get("probabilidad_coautoria")
        umbral = metodo.get("umbral_aplicado")
        if umbral is None:
            umbral = umbral_general
        metodo["umbral_aplicado"] = umbral

        if p is not None and umbral is not None:
            metodo["clasificacion_por_umbral"] = "favorable" if p >= umbral else "desfavorable"
        else:
            metodo["clasificacion_por_umbral"] = None

    instruccion_idioma = datos["idiomas"]["instruccion_por_idioma"][idioma]
    return {
        "model": etiqueta,
        "messages": [
            {"role": "system", "content": instrucciones + "\n" + instruccion_idioma},
            {"role": "user", "content": json.dumps(entrada, ensure_ascii=False, indent=2)}
        ],
        "stream": False,
        "think": False,
        "options": {"temperature": 0, "num_ctx": num_ctx, "num_predict": num_predict}
    }

## 3. Consultar a Ollama

Se guarda la respuesta, el tiempo y el estado de cada intento. Si un modelo no acepta `think=False`, se reintenta sin ese campo.

In [3]:
def pedir_justificacion(solicitud):
    inicio = time.time()
    enviada = solicitud
    resultado = {"estado": "error", "respuesta": "", "error": ""}

    try:
        r = requests.post(url_ollama + "/chat", json=enviada, timeout=1800)

        # Si el modelo no acepta 'think', reintentar sin ese campo
        if r.status_code == 400 and "think" in r.text.lower():
            enviada = solicitud.copy()
            enviada.pop("think")
            r = requests.post(url_ollama + "/chat", json=enviada, timeout=1800)

        r.raise_for_status()
        salida = r.json()
        texto = salida.get("message", {}).get("content", "")
        resultado["respuesta"] = texto

        if salida.get("done_reason") == "length":
            resultado["estado"] = "truncada"
        elif salida.get("done") and texto.strip() != "":
            resultado["estado"] = "completa"
        else:
            resultado["estado"] = "incompleta"

    except Exception as e:
        resultado["error"] = str(e)

    resultado["tiempo_s"] = round(time.time() - inicio, 2)
    return resultado, enviada

## 4. Ejecutar las 96 pruebas

Se ejecuta un modelo a la vez, se guarda cada respuesta al terminar y se omiten las que ya estén completas.

In [4]:
# Revisar qué modelos están instalados en Ollama
r = requests.get(url_ollama + "/tags", timeout=15)
r.raise_for_status()
instalados = [m["name"].lower() for m in r.json()["models"]]

faltantes = []
nuevas = 0

for modelo in modelos_a_probar:
    id_modelo = modelo["id_modelo"]
    etiqueta = modelo["etiqueta_ollama"]
    nombre = modelo["nombre_modelo"]

    if etiqueta.lower() not in instalados:
        faltantes.append(etiqueta)
        print("NO INSTALADO:", id_modelo, "|", etiqueta)
        continue

    print(f"\nMODELO: {id_modelo} | {nombre}")

    for id_caso in ids_casos:
        for idioma in idiomas:
            if (id_modelo, id_caso, idioma) in completadas:
                print("  ", id_caso, idioma, "| ya completada")
                continue

            # Contar intentos previos para este modelo, caso e idioma
            intento = 1
            for f in registros:
                if f["id_modelo"] == id_modelo and f["id_caso"] == id_caso and f["idioma"] == idioma:
                    intento += 1

            solicitud = preparar_solicitud(casos[id_caso], idioma, etiqueta)
            resultado, enviada = pedir_justificacion(solicitud)

            fila = {
                "id_caso": id_caso,
                "id_modelo": id_modelo,
                "idioma": idioma,
                "intento": intento,
                "solicitud_json": json.dumps(enviada, ensure_ascii=False),
                "estado": resultado["estado"],
                "tiempo_s": resultado["tiempo_s"],
                "respuesta": resultado["respuesta"],
                "error": resultado["error"],
                "evaluacion": "",
                "motivo_evaluacion": ""
            }

            with open(archivo_resultados, "a", encoding="utf-8", newline="") as f:
                csv.DictWriter(f, fieldnames=columnas).writerow(fila)

            registros.append(fila)
            if fila["estado"] == "completa":
                completadas.add((id_modelo, id_caso, idioma))
            nuevas += 1

            print("  ", id_caso, idioma, "|", fila["estado"], "|", fila["tiempo_s"], "s")
            if fila["error"]:
                print("     Error:", fila["error"][:180])

    # Liberar el modelo de memoria antes de cargar el siguiente
    try:
        requests.post(url_ollama + "/generate", json={"model": etiqueta, "keep_alive": 0}, timeout=60)
    except Exception:
        pass

print("\nIntentos nuevos guardados:", nuevas)
print("Total de filas en resultados.csv:", len(registros))
if faltantes:
    print("\nModelos pendientes de instalar:")
    for etiqueta in faltantes:
        print("  ollama pull", etiqueta)


MODELO: M14 | Command R7B 12-2024
   C001 es | completa | 12.18 s
   C001 en | completa | 5.63 s
   C002 es | completa | 5.15 s
   C002 en | completa | 5.26 s
   C003 es | completa | 5.83 s
   C003 en | completa | 5.62 s
   C004 es | completa | 4.68 s
   C004 en | completa | 5.14 s
   C005 es | completa | 5.77 s
   C005 en | completa | 5.67 s
   C006 es | completa | 4.67 s
   C006 en | completa | 5.43 s
   C007 es | completa | 5.28 s
   C007 en | completa | 5.65 s
   C008 es | completa | 5.32 s
   C008 en | completa | 5.87 s

MODELO: M22 | LFM2.5 8B A1B
   C001 es | completa | 13.97 s
   C001 en | completa | 6.45 s
   C002 es | completa | 6.2 s
   C002 en | completa | 8.99 s
   C003 es | completa | 8.29 s
   C003 en | completa | 6.75 s
   C004 es | completa | 8.52 s
   C004 en | completa | 8.02 s
   C005 es | completa | 9.87 s
   C005 en | completa | 9.8 s
   C006 es | completa | 8.26 s
   C006 en | completa | 6.22 s
   C007 es | completa | 11.15 s
   C007 en | completa | 7.76 s
   C0

## 5. Revisar el avance

In [5]:
completas = 0
for m in modelos_a_probar:
    for id_caso in ids_casos:
        for idioma in idiomas:
            if (m["id_modelo"], id_caso, idioma) in completadas:
                completas += 1

total = len(modelos_a_probar) * len(ids_casos) * len(idiomas)
print(f"Avance: {completas} de {total} respuestas completas")
print("Pendientes:", total - completas)
print("Archivo:", archivo_resultados)

Avance: 96 de 96 respuestas completas
Pendientes: 0
Archivo: resultados.csv
